# Week 12: the 2D DWT of images

**What we do.** We decompose a photograph with the two-dimensional discrete wavelet transform, look at the subbands of one level in detail, compare decomposition levels and the filters db1, db3 and db5, check perfect reconstruction, and measure how sparse the image becomes in the wavelet basis.

**What to expect.** One level gives four quarter-size subbands: an approximation (a smaller copy of the image) and three detail subbands showing horizontal, vertical and diagonal edges. Most coefficients are close to zero; only edges produce large details. Longer filters put more energy into the approximation.

**How to read the code.** Functions come from `wavelets_lab.dwt` (2D DWT, PyWavelets underneath, periodic boundary) and `wavelets_lab.images` (loading and display). The coefficients are a list `[cA_J, (cH_J, cV_J, cD_J), …, (cH_1, cV_1, cD_1)]`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelets_lab import dwt, images

## One level: four subbands

**What we do.** We compute one level of the db3 transform of `camera` and show the four subbands separately.

**What to expect.** `cA_1` (256×256) looks like the photograph at half resolution. `cH_1` responds to horizontal edges (changes from top to bottom), `cV_1` to vertical edges (the tripod legs), `cD_1` to diagonal structures; elsewhere the details are close to zero (grey in the scaled display).

**How to read the code.** `dwt.decompose_image(camera, "db3", 1)` returns `[cA_1, (cH_1, cV_1, cD_1)]`; `coefficients[1]` is the tuple of details, unpacked into three variables. Details have both signs, so they are shown with `scaled=True`; the axes count coefficients, not pixels.

In [ ]:
camera = images.load_test_image("camera")
coefficients = dwt.decompose_image(camera, "db3", 1)
c_a = coefficients[0]
c_h, c_v, c_d = coefficients[1]
labels = {"xlabel": "coefficient column", "ylabel": "coefficient row"}
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
images.show_image(c_a, "approximation cA_1", ax=axes[0], scaled=True, **labels)
images.show_image(c_h, "horizontal details cH_1", ax=axes[1], scaled=True, **labels)
images.show_image(c_v, "vertical details cV_1", ax=axes[2], scaled=True, **labels)
images.show_image(c_d, "diagonal details cD_1", ax=axes[3], scaled=True, **labels)
plt.tight_layout()
plt.show()

## Several levels in one picture

**What we do.** We arrange all subbands of 1, 2 and 3 levels in the classical layout.

**What to expect.** The top-left corner holds the approximation, which gets smaller with each level; around it are the detail subbands of the levels, the finest ones (largest) at the outside. Every subband is rescaled separately to make the details visible.

**How to read the code.** `dwt.coefficient_image(coefficients)` glues the subbands into one 512×512 array (absolute values, each subband scaled to `[0, 255]`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, level in zip(axes, (1, 2, 3), strict=True):
    layout = dwt.coefficient_image(dwt.decompose_image(camera, "db3", level))
    images.show_image(layout, f"db3, {level} level(s)", ax=ax, **labels)
plt.tight_layout()
plt.show()

## Perfect reconstruction

**What we do.** We decompose with db5 at 4 levels and reconstruct.

**What to expect.** The largest error is about `1e-12`: the transform is invertible.

**How to read the code.** `dwt.reconstruct_image(coefficients, "db5")` is the inverse transform.

In [ ]:
coefficients = dwt.decompose_image(camera, "db5", 4)
np.abs(dwt.reconstruct_image(coefficients, "db5") - camera).max()

## Sparsity: most coefficients are small

**What we do.** We sort the absolute values of all wavelet coefficients (3 levels, db3) from largest to smallest and compare with the sorted pixel values.

**What to expect.** The pixel values decrease slowly: all pixels matter. The wavelet coefficients drop by several orders of magnitude: a few percent of the coefficients carry almost all the information, so the rest can be set to zero (compression, next week).

**How to read the code.** `np.concatenate` with `np.ravel` collects all coefficients into one vector; `np.sort(...)[::-1]` sorts in decreasing order. The horizontal axis is the rank divided by the number of values (from 0 = largest to 1 = smallest); the vertical axis is logarithmic.

In [ ]:
coefficients = dwt.decompose_image(camera, "db3", 3)
values = np.concatenate(
    [np.ravel(coefficients[0])]
    + [np.ravel(band) for bands in coefficients[1:] for band in bands]
)
rank = np.arange(values.size) / values.size
plt.figure(figsize=(8, 4))
plt.semilogy(
    rank, np.sort(np.abs(values))[::-1], label="wavelet coefficients (db3, 3 levels)"
)
plt.semilogy(rank, np.sort(camera.ravel())[::-1], label="pixel values")
plt.xlabel("rank / number of values")
plt.ylabel("absolute value (log scale)")
plt.ylim(1e-2, 1e4)
plt.legend()
plt.show()

## How much energy stays in the approximation?

**What we do.** For db1, db3 and db5 (3 levels) we compute the share of the total energy contained in the approximation `cA_3`, which has only 1/64 of all coefficients.

**What to expect.** In every case about 98.5% of the energy in 1.6% of the coefficients; the share grows slightly with the filter length because longer filters with more vanishing moments represent smooth regions better.

**How to read the code.** The energy of an array is `Σ values²`; for an orthogonal transform the total energy of the coefficients equals the energy of the image, so we divide by `np.sum(camera**2)`.

In [ ]:
for name in ("db1", "db3", "db5"):
    coefficients = dwt.decompose_image(camera, name, 3)
    share = np.sum(coefficients[0] ** 2) / np.sum(camera**2)
    print(f"{name}: {share:.4%} of the energy in cA_3 (1/64 of the coefficients)")

## Reference: scripts and functions used in this notebook

**What we do.** We list every function of the course package used above with its purpose, input, output and a recipe for writing it yourself.

**What to expect.** Two modules are used: `wavelets_lab.images` (loading and display) and `wavelets_lab.dwt` (the 2D transform).

**How to read the code.** The image sides must be divisible by `2**level`; the periodic transform keeps the total number of coefficients equal to the number of pixels.

**Library functions called directly in this notebook.**

| Function | What it does | Input → output |
|---|---|---|
| `np.ravel(a)` | flattens to 1D | array → 1D array |
| `np.concatenate(list)` | joins 1D arrays | list of arrays → array |
| `np.sort(a)[::-1]` | sorts in decreasing order | array → array |
| `plt.semilogy`, `plt.ylim` | logarithmic vertical axis and its range | – |

### Module `wavelets_lab.images` (file `src/wavelets_lab/images.py`)

Images as NumPy arrays and simple pixel operations. **Convention used everywhere in the course:** a greyscale image is a 2D array of type `float64` with values on the scale `[0, 255]` (0 = black, 255 = white); a colour image is a 3D array of shape `(rows, cols, 3)` on the same scale. Working in `float64` (instead of the `uint8` stored in image files) avoids overflow and rounding during computations.

The module starts with the imports
```python
from pathlib import Path
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
```
and the constant `MAX_INTENSITY = 255.0`.

#### `images.load_test_image(name="camera")`

**What it is.** Loads one of the licence-free test images shipped with scikit-image (`skimage.data`), so that no image file is needed.

**Input.**
- `name` (`str`, default `"camera"`): name of a function in `skimage.data`, e.g. `"camera"` (512×512 greyscale), `"astronaut"` (512×512 RGB), `"coins"`, `"checkerboard"`.

**Output.** `numpy.ndarray` of type `float64` on the scale `[0, 255]` (2D for greyscale, `(rows, cols, 3)` for colour). Raises `ValueError` if the name does not exist.

**How to write it yourself.**
1. Import the module of test images: `from skimage import data`.
2. Look the loader up by name without failing: `loader = getattr(data, name, None)`.
3. If `loader` is `None` or not callable (`callable(loader)` is `False`), raise `ValueError(f"skimage.data has no test image called {name!r}")`.
4. Call it: `raw = loader()` returns a `uint8` array.
5. Return `to_float(raw)` (see `images.to_float`), which converts to `float64` on `[0, 255]`.

#### `images.to_float(image)` (used internally)

**What it is.** Converts any image array to the course convention: `float64` on `[0, 255]`.

**Input.**
- `image` (array-like): `uint8`, `bool`, integer or floating-point image.

**Output.** `numpy.ndarray` of type `float64`.

**How to write it yourself.**
1. `array = np.asarray(image)`.
2. If `array.dtype == np.bool_`: return `array.astype(np.float64) * 255.0` (True becomes white).
3. If the dtype is floating (`np.issubdtype(array.dtype, np.floating)`), the array is not empty and all values lie in `[0, 1]` (`array.min() >= 0` and `array.max() <= 1`): the image uses the unit scale, so return `array.astype(np.float64) * 255.0`.
4. Otherwise (integers, or floats already on a large scale) return `array.astype(np.float64)`.

#### `images.show_image(image, title=None, ax=None, scaled=False, xlabel="column [pixel]", ylabel="row [pixel]")`

**What it is.** Displays a greyscale or colour image with Matplotlib, using a 256-level grey colour map and **labelled axes** (by default pixel coordinates: column on the horizontal axis, row on the vertical axis, with row 0 at the top).

**Input.**
- `image` (array-like): 2D greyscale or `(rows, cols, 3)` RGB image on `[0, 255]`.
- `title` (`str` or `None`): title of the plot.
- `ax` (`matplotlib.axes.Axes` or `None`): where to draw; when `None` a new figure is created. Passing `ax` lets you place several images in one figure (`fig, axes = plt.subplots(1, 3)`).
- `scaled` (`bool`, default `False`): `False` maps 0 to black and 255 to white (true intensities); `True` stretches the smallest value of the data to black and the largest to white, like Matlab's `imagesc`. Use `True` for arrays that are not intensities (spectra, gradients, phases).
- `xlabel`, `ylabel` (`str`): axis labels; change them when the axes mean something else (e.g. angle and offset for a sinogram).

**Output.** The `Axes` object that contains the image (useful for further changes such as `ax.set_xlim`).

**How to write it yourself.**
1. `array = np.asarray(image, dtype=np.float64)`.
2. If `ax is None`: `_, ax = plt.subplots()`.
3. If `array.ndim == 3` (colour): `ax.imshow(to_uint8(array))`; Matplotlib then uses the RGB values directly.
4. Otherwise (greyscale): choose limits `vmin, vmax = (None, None)` if `scaled` else `(0.0, 255.0)` and call `ax.imshow(array, cmap=plt.get_cmap("gray", 256), vmin=vmin, vmax=vmax)`. With `vmin=vmax=None` Matplotlib uses the data minimum and maximum.
5. If a title is given: `ax.set_title(title)`.
6. Label both axes: `ax.set_xlabel(xlabel)` and `ax.set_ylabel(ylabel)`.
7. Return `ax`.

#### `images.to_uint8(image)` (used internally)

**What it is.** The inverse direction: prepares an image for saving or for `imshow` of colour images, which expects bytes.

**Input.**
- `image` (array-like): image on the scale `[0, 255]`, possibly with values outside it.

**Output.** `numpy.ndarray` of type `uint8`.

**How to write it yourself.**
1. Convert to floats: `np.asarray(image, dtype=np.float64)`.
2. Round to the nearest integer with `np.rint`.
3. Clip to the valid range with `np.clip(..., 0, 255)`; without this, e.g. 260 would wrap around to 4.
4. Cast with `.astype(np.uint8)` and return.

### Module `wavelets_lab.dwt` (file `src/wavelets_lab/dwt.py`)

The Mallat algorithm (discrete wavelet transform, DWT) in 1D, written from scratch, and the 2D DWT of images via PyWavelets. Signals are treated as **periodic**, which makes the transform orthogonal (it preserves energy) and matches `mode="periodization"` of PyWavelets; then a signal of length `N` gives `N/2` approximation and `N/2` detail coefficients. Imports:
```python
import numpy as np
import pywt
```
Two internal helpers: `_decomposition_filters(wavelet)` returns `pywt.Wavelet(wavelet).dec_lo` and `.dec_hi` as arrays (`dec_lo` is `rec_lo` reversed); `_analysis_indices(n, length)` returns the integer matrix `idx[k, m] = (2k + length // 2 - m) mod n` for `k = 0, …, n/2 - 1` and `m = 0, …, length - 1`, i.e. which sample is multiplied by filter tap `m` in output `k`.

#### `dwt.decompose_image(image, wavelet="db2", level=3)`

**What it is.** The 2D DWT of an image with `pywt.wavedec2` (periodization). One level filters the rows and then the columns, giving four subbands: the approximation `cA` (low-pass in both directions) and the details `cH` (horizontal edges), `cV` (vertical edges) and `cD` (diagonal); the next level decomposes `cA` again.

**Input.** `image` (array-like): greyscale image with sides divisible by `2**level`; `wavelet` (`str`), e.g. `"db1"`, `"db3"`, `"db5"`; `level` (`int`).

**Output.** List `[cA_J, (cH_J, cV_J, cD_J), …, (cH_1, cV_1, cD_1)]`.

**How to write it yourself.**
1. Call `pywt.wavedec2(np.asarray(image, dtype=np.float64), wavelet, mode="periodization", level=level)`.
2. Return the result as a list.

#### `dwt.reconstruct_image(coefficients, wavelet="db2")`

**What it is.** The inverse 2D DWT (`pywt.waverec2` with periodization).

**Input.** `coefficients` (list in the format of `decompose_image`); `wavelet` (`str`).

**Output.** `float64` image.

**How to write it yourself.** Return `np.asarray(pywt.waverec2(coefficients, wavelet, mode="periodization"), dtype=np.float64)`.

#### `dwt.coefficient_image(coefficients)`

**What it is.** Arranges all subbands of a 2D DWT in one image of the original size, in the classical layout (approximation in the top-left corner, details of each level around it). Each subband's absolute values are rescaled separately to `[0, 255]`; otherwise the huge approximation would make all details look black.

**Input.** `coefficients` (list in the format of `decompose_image`).

**Output.** `float64` image of the size of the transformed image.

**How to write it yourself.**
1. Write a helper `rescale(band)`: take `np.abs(band)`; if its maximum `top > 0` return `values * 255 / top`, else return the values.
2. Apply it to `coefficients[0]` and to each of the three bands in every detail tuple, keeping the list structure.
3. Glue everything with `array, _ = pywt.coeffs_to_array(scaled)` and return `array`.